In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [ ]:
params = ["CLint", "KbBR", "KbMU", "KbBO", "KbAD", "KbRB"]

df_nlmixr2 = pd.read_csv("nlmixr2_map_MI.csv")
df_nlmixr2 = df_nlmixr2[["ID"] + params].copy()
df_nlmixr2["algo"] = "nlmixr2"

df_sbml = pd.read_csv("Sbml_map_MI.csv")
if "id_ref" in df_sbml.columns:
    df_sbml = df_sbml.rename(columns={"id_ref": "ID"})
df_sbml = df_sbml[["ID"] + params].copy()
df_sbml["algo"] = "Sbml"

df_simwork = pd.read_csv("Simwork_map_MI.csv")
if "id_ref" in df_simwork.columns:
    df_simwork = df_simwork.rename(columns={"id_ref": "ID"})
df_simwork = df_simwork[["ID"] + params].copy()
df_simwork["algo"] = "Simwork"

map_df = pd.concat([df_nlmixr2, df_sbml, df_simwork], ignore_index=True)
map_obs = map_df.melt(id_vars=["ID", "algo"], value_vars=params, var_name="descriptor").reset_index(drop=True)
map_obs["Source"] = map_obs["algo"]


In [ ]:
sns.set_theme(style="whitegrid")
palette = {"nlmixr2": "#F8766D", "Sbml": "#00BFC4", "Simwork": "#619CFF"} 

fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(15, 10))
fig.suptitle("Individual parameter estimates", fontsize=18, y=1.02)
axes = axes.flatten()
for i, param in enumerate(params):
    ax = axes[i]
    df_param = map_obs[map_obs["descriptor"] == param]
    
    if param == "CLint":
        sns.histplot(
            data=df_param, x="value", hue="Source", 
            bins=30, alpha=0.5, edgecolor=None, ax=ax,
            palette=palette, legend=True
        )
        ax.set_title("CLint", fontsize=14)
        ax.set_xlabel("Parameter value")
        ax.set_ylabel("Count")
    else:
        sns.barplot(
            data=df_param, x="Source", y="value", hue="Source",
            ax=ax, alpha=0.8, palette=palette, dodge=False
        )
        ax.set_title(param, fontsize=14)
        ax.set_xlabel("")
        ax.set_ylabel("Parameter value")
        
        ax.tick_params(axis='x', rotation=45) 
        
        if ax.get_legend() is not None:
            ax.get_legend().remove()

plt.tight_layout()
plt.show()

In [ ]:
VPC_nlmixr2 = pd.read_csv("nlmixr2_VPC.csv")
VPC_Sbml    = pd.read_csv("Sbml_VPC.csv")
VPC_Simwork = pd.read_csv("Simwork_VPC.csv")

VPC_Sbml["bin_center"] = VPC_Sbml["bin_center"] / 3600.0
VPC_Simwork["bin_center"] = VPC_Simwork["bin_center"] / 3600.0

VPC_nlmixr2["Source"] = "nlmixr2"
VPC_Sbml["Source"] = "Sbml"
VPC_Simwork["Source"] = "Simwork"

vpc_data = pd.concat([VPC_nlmixr2, VPC_Sbml, VPC_Simwork], ignore_index=True)

cols_to_exp = ["q_obs", "pred_median", "pred_lower", "pred_upper"]
for c in cols_to_exp:
    vpc_data[c] = np.exp(vpc_data[c])

In [ ]:
fig2, ax2 = plt.subplots(figsize=(10, 6))

quantiles = vpc_data["quantile"].unique()
sources = ["nlmixr2", "Sbml", "Simwork"]

for source in sources:
    df_source = vpc_data[vpc_data["Source"] == source]
    color = palette[source]

    for q in quantiles:
        df_q = df_source[df_source["quantile"] == q].sort_values("bin_center")

        ax2.fill_between(
            df_q["bin_center"],
            df_q["pred_lower"],
            df_q["pred_upper"],
            color=color,
            alpha=0.35,  
        )

        label = source if q == quantiles[0] else ""
        ax2.plot(
            df_q["bin_center"],
            df_q["pred_median"],
            color=color,
            linewidth=2,
            label=label,
        )

obs_data = vpc_data[vpc_data["Source"] == "nlmixr2"]
for q in quantiles:
    df_obs_q = obs_data[obs_data["quantile"] == q].sort_values("bin_center")
    label = "Observed" if q == quantiles[0] else ""
    ax2.plot(
        df_obs_q["bin_center"],
        df_obs_q["q_obs"],
        color="black",
        linestyle="dashed",
        linewidth=2,
        label=label,
    )

ax2.set_yscale("log")
ax2.set_xlabel("Time (h)", fontsize=12)
ax2.set_ylabel("Mavoglurant concentration (ng/mL)", fontsize=12)
ax2.set_title("Visual predictive checks", fontsize=16, pad=15)

handles, labels = ax2.get_legend_handles_labels()
by_label = dict(zip(labels, handles))
ax2.legend(by_label.values(), by_label.keys(), title="Source", loc="best")

fig2.tight_layout()
plt.show()